# Phase 4C.3 Lab - Specialist Pricing Boundary

Mục tiêu: hiểu Modal Specialist adapter và ensemble fallback behavior.

Default expected output: focused tests pass without Modal dependency/auth.

Safety: Specialist smoke chỉ chạy khi real model mode và Modal config đã set.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_v3_root(start: str | None = None) -> Path:
    path = Path(start or os.getcwd()).resolve()
    for candidate in (path, *path.parents):
        if candidate.name == "shopping_assistant_v3" and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the tech2ai/shopping_assistant_v3 tree.")


V3_ROOT = find_v3_root()
os.chdir(V3_ROOT)
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))


def run(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess[str] | None:
    print("$ " + " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=V3_ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds.")
        return None

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"exit_code={result.returncode}")
    return result


print(f"V3_ROOT={V3_ROOT}")
print("Default safety flags:")
for name in ("ENABLE_REAL_SEARCH", "ENABLE_REAL_MODEL_CALLS", "ENABLE_AGENTS_SDK"):
    print(f"{name}={os.getenv(name, '<unset>')}")


## 1. Chạy Specialist boundary tests

Command này kiểm tra missing config, missing dependency, sanitized errors và
real estimator assembly/fallback.


In [ ]:
run(["uv", "run", "pytest", "tests/test_real_pricing.py", "tests/test_real_pricing_specialist.py", "-q", "--tb=short"], timeout=180)


## 2. Kiểm tra Modal env readiness mà không in secrets


In [ ]:
for key in ("ENABLE_REAL_MODEL_CALLS", "PRICER_SPECIALIST_SERVICE", "PRICER_SPECIALIST_CLASS"):
    value = os.getenv(key, "")
    print(f"{key}={value or '<unset>'}")


## 3. Opt-in Specialist smoke cell

Only runs when env is configured. This may call a Modal remote service.


In [ ]:
specialist_ready = (
    os.getenv("ENABLE_REAL_MODEL_CALLS", "").strip().lower() == "true"
    and os.getenv("PRICER_SPECIALIST_SERVICE")
    and os.getenv("PRICER_SPECIALIST_CLASS")
)
if specialist_ready:
    run(["uv", "run", "pytest", "tests/test_real_pricing_specialist_smoke.py", "-q", "--tb=short"], timeout=300)
else:
    print("Skipped Specialist smoke. Requires ENABLE_REAL_MODEL_CALLS=true plus PRICER_SPECIALIST_SERVICE and PRICER_SPECIALIST_CLASS.")


## 4. Cách đọc kết quả

- Boundary tests pass: adapter/fallback logic đúng.
- Smoke skipped: expected khi Modal chưa cấu hình.
- Full ensemble chỉ thật sự chạy đủ khi Frontier, Specialist, và Neural đều
  available.
- Không paste Modal tokens hoặc private service logs.
